### Kaggle Setup

In [ ]:
!pip install transformers datasets evaluate seqeval scikit-learn optimum[onnxruntime] optuna

In [ ]:
import os
from pathlib import Path

# NER_DATASET: Kaggle input path when dataset is attached
# Verify after attaching: os.listdir("/kaggle/input/datasets/maazahmad69/")
NER_DATASET = "/kaggle/input/datasets/maazahmad69/distilbert-ner-smart-stock"

# Set to checkpoint path to resume, or None for fresh training
RESUME_FROM = f"{NER_DATASET}/distilbert-ner-smart-stock/checkpoint-1365"
# RESUME_FROM = None  # uncomment for fresh training

In [ ]:
# ── Label map (define FIRST, before any loaders) ──────────────────────────────
LABEL2ID = {"O": 0, "B-FOOD": 1, "I-FOOD": 2, "B-QTY": 3, "B-UNIT": 4, "B-PRICE": 5}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

### **CORD LOADER**
#### CORD's ground_truth is a JSON string. The valid_line field contains per-word annotations

In [ ]:
import json
from datasets import load_dataset

# Module-level — must be defined before calling any loader or build function
CORD_TO_ENTITY = {
    "menu.nm":        "FOOD",
    "menu.sub_nm":    "FOOD",
    "menu.cnt":       "QTY",
    "menu.unitprice": "PRICE",
    "menu.price":     "PRICE",
}

def load_cord_ner() -> list[dict]:
    """
    Extract word-level NER annotations from CORD valid_line.
    Groups words by group_id into logical receipt lines.
    Returns list of {"tokens": [...], "ner_tags": [...]} dicts.
    """
    cord = load_dataset("naver-clova-ix/cord-v2")
    examples = []

    for split in ["train", "validation", "test"]:
        for sample in cord[split]:
            try:
                data = json.loads(sample["ground_truth"])
            except (json.JSONDecodeError, KeyError):
                continue

            valid_lines = data.get("valid_line", [])

            # Group by group_id — same group = same receipt line
            from collections import defaultdict
            groups = defaultdict(list)
            for line in valid_lines:
                groups[line["group_id"]].append(line)

            for gid, lines in groups.items():
                tokens, tags = [], []
                prev_entity = None

                for line in lines:
                    # category is on the line object — shared by all words in the line
                    category = line.get("category", "")
                    entity = CORD_TO_ENTITY.get(category, None)

                    for word in line.get("words", []):
                        text = word.get("text", "").strip()
                        if not text:
                            continue

                        if entity is None:
                            tags.append(LABEL2ID["O"])
                            prev_entity = None
                        elif entity == "FOOD" and entity == prev_entity:
                            # Only FOOD spans get I- continuation prefix
                            # QTY/UNIT/PRICE are always single-token → always B-
                            tags.append(LABEL2ID["I-FOOD"])
                        else:
                            tags.append(LABEL2ID[f"B-{entity}"])
                            prev_entity = entity

                        tokens.append(text)

                if tokens and any(t != LABEL2ID["O"] for t in tags):
                    examples.append({"tokens": tokens, "ner_tags": tags})

    print(f"CORD: {len(examples)} annotated lines")
    return examples

Expected yield: ~11,000 lines with at least one non-O entity.


### **TASTEset Loader**

In [ ]:
def load_tasteset_ner() -> list[dict]:
    """
    Load TASTEset and remap entity tags to Smart-Stock schema.
    TASTEset is pre-tokenized — use 'recipes' field as tokens,
    'ner_tags' field for labels.
    """
    TASTESET_MAP = {
        "B-FOOD":     "B-FOOD",
        "I-FOOD":     "I-FOOD",
        "B-QUANTITY": "B-QTY",
        "I-QUANTITY": "O",    # collapse multi-token quantities to first only
        "B-UNIT":     "B-UNIT",
        "I-UNIT":     "O",    # collapse multi-token units to first only
        # everything else → O
    }

    tasteset = load_dataset("dmargutierrez/TASTESet")
    examples = []

    for split in ["train", "test"]:
        for sample in tasteset[split]:
            tokens = sample["recipes"]
            raw_tags = sample["ner_tags"]

            if not tokens or not raw_tags:
                continue

            tags = [
                LABEL2ID.get(TASTESET_MAP.get(t, "O"), LABEL2ID["O"])
                for t in raw_tags[:len(tokens)]
            ]

            if any(t != LABEL2ID["O"] for t in tags):
                examples.append({"tokens": tokens, "ner_tags": tags})

    print(f"TASTEset: {len(examples)} annotated lines")
    return examples

Expected yield: ~680 examples (490 train + 210 test after filtering empty).


### **Synthetic Grocery Lines**
Generates English grocery receipt abbreviations with ground-truth annotations. Fills the gap between recipe language and receipt abbreviations.

In [ ]:
import random

# ── Food vocabulary ───────────────────────────────────────────────────────────

PRODUCE = [
    ("Strawberries", "STRWBRY"), ("Blueberries", "BLUBRY"),
    ("Raspberries", "RSPBRY"), ("Blackberries", "BLKBRY"),
    ("Bananas", "BANANA"), ("Apples", "APPLE"),
    ("Green Apples", "GRN APPLE"), ("Red Apples", "RED APPLE"),
    ("Gala Apples", "GALA APPLE"), ("Fuji Apples", "FUJI APPLE"),
    ("Oranges", "ORANGE"), ("Lemons", "LEMON"), ("Limes", "LIME"),
    ("Grapes", "GRAPES"), ("Watermelon", "WTRMLN"),
    ("Cantaloupe", "CNTLOPE"), ("Pineapple", "PNAPPL"),
    ("Mango", "MANGO"), ("Peaches", "PEACH"), ("Pears", "PEAR"),
    ("Avocado", "AVOCDO"), ("Tomatoes", "TOMATO"),
    ("Cherry Tomatoes", "CHRRY TOM"), ("Cucumbers", "CUCMBR"),
    ("Carrots", "CARROT"), ("Celery", "CELERY"),
    ("Broccoli", "BROCC"), ("Cauliflower", "CAULFLWR"),
    ("Bell Peppers", "BLL PEPR"), ("Jalapenos", "JALPNO"),
    ("Onions", "ONION"), ("Red Onions", "RED ONION"),
    ("Green Onions", "GRN ONION"), ("Garlic", "GARLIC"),
    ("Potatoes", "POTATO"), ("Sweet Potatoes", "SWT POTATO"),
    ("Russet Potatoes", "RSST POT"), ("Baby Spinach", "BBY SPNCH"),
    ("Spinach", "SPNCH"), ("Kale", "KALE"),
    ("Romaine Lettuce", "ROMAINE"), ("Iceberg Lettuce", "ICBRG LET"),
    ("Cabbage", "CABBAGE"), ("Mushrooms", "MSHRM"),
    ("Green Beans", "GRN BEAN"), ("Corn", "CORN"),
    ("Zucchini", "ZUCCHNI"), ("Eggplant", "EGGPLNT"),
]

DAIRY = [
    ("Whole Milk", "WHL MLK"), ("2 Percent Milk", "2PCT MLK"),
    ("Skim Milk", "SKM MLK"), ("Almond Milk", "ALMD MLK"),
    ("Oat Milk", "OAT MLK"), ("Soy Milk", "SOY MLK"),
    ("Greek Yogurt", "GRK YGRT"), ("Vanilla Yogurt", "VNLA YGRT"),
    ("Plain Yogurt", "PLN YGRT"), ("Cheddar Cheese", "CHDR CH"),
    ("Mozzarella", "MOZZ"), ("Swiss Cheese", "SWISS CH"),
    ("Parmesan", "PARM"), ("Cream Cheese", "CRM CH"),
    ("Butter", "BTTR"), ("Sour Cream", "SOR CRM"),
    ("Heavy Cream", "HVY CRM"), ("Cottage Cheese", "CTTG CH"),
    ("Organic Eggs", "ORG EGGS"), ("Large Eggs", "LRG EGGS"),
    ("Extra Large Eggs", "XLG EGGS"), ("Free Range Eggs", "FR RNG EGG"),
]

MEAT = [
    ("Chicken Breast", "CHKN BRST"), ("Chicken Thighs", "CHKN THGH"),
    ("Chicken Wings", "CHKN WING"), ("Whole Chicken", "WHL CHKN"),
    ("Ground Chicken", "GRD CHKN"), ("Ground Beef", "GRD BEEF"),
    ("Beef Steak", "BEEF STK"), ("Sirloin Steak", "SRLN STK"),
    ("Ribeye Steak", "RBEY STK"), ("Pork Chops", "PRK CHPS"),
    ("Ground Pork", "GRD PRK"), ("Bacon", "BACON"),
    ("Turkey Breast", "TRKY BRST"), ("Ground Turkey", "GRD TRKY"),
    ("Ham", "HAM"), ("Sausage", "SAUSAGE"),
    ("Italian Sausage", "ITAL SAU"), ("Beef Ribs", "BEEF RIBS"),
    ("Lamb Chops", "LMB CHPS"), ("Beef Mince", "BEEF MNCE"),
]

SEAFOOD = [
    ("Salmon Fillet", "SLMN FLT"), ("Tilapia Fillet", "TLPIA FLT"),
    ("Cod Fillet", "COD FLT"), ("Shrimp", "SHRIMP"),
    ("Tuna Steak", "TUNA STK"), ("Crab Meat", "CRAB MEAT"),
    ("Canned Tuna", "CND TUNA"), ("Lobster Tail", "LBSTR TL"),
    ("Scallops", "SCALLOP"), ("Mussels", "MUSSELS"),
]

BAKERY = [
    ("White Bread", "WHT BRD"), ("Whole Wheat Bread", "WHWHT BRD"),
    ("Sourdough Bread", "SRDGH BRD"), ("Bagels", "BAGELS"),
    ("English Muffins", "ENG MFFN"), ("Hamburger Buns", "HMBRGR BN"),
    ("Hot Dog Buns", "HOTDOG BN"), ("Croissants", "CROISSNT"),
    ("Flour Tortillas", "FLR TRTLA"), ("Corn Tortillas", "CRN TRTLA"),
    ("Pita Bread", "PITA BRD"), ("Naan", "NAAN"),
    ("Dinner Rolls", "DNR ROLLS"), ("Baguette", "BAGUETTE"),
]

PANTRY = [
    ("Brown Rice", "BRN RICE"), ("White Rice", "WHT RICE"),
    ("Jasmine Rice", "JSMN RICE"), ("Basmati Rice", "BSMTI RICE"),
    ("Pasta", "PASTA"), ("Spaghetti", "SPGHETTI"),
    ("Macaroni", "MACARONI"), ("Penne Pasta", "PENNE"),
    ("Pasta Sauce", "PST SCE"), ("Marinara Sauce", "MRNRA"),
    ("Peanut Butter", "PNT BTTR"), ("Almond Butter", "ALMD BTTR"),
    ("Jelly", "JELLY"), ("Strawberry Jam", "STRWBRY JAM"),
    ("Black Beans", "BLK BEAN"), ("Kidney Beans", "KDNY BEAN"),
    ("Chickpeas", "CHCKPEA"), ("Lentils", "LENTILS"),
    ("Canned Corn", "CND CORN"), ("Canned Tomatoes", "CND TOM"),
    ("Olive Oil", "OLV OIL"), ("Vegetable Oil", "VEG OIL"),
    ("Coconut Oil", "CCN OIL"), ("Flour", "FLOUR"),
    ("Sugar", "SUGAR"), ("Brown Sugar", "BRN SUGAR"),
    ("Honey", "HONEY"), ("Salt", "SALT"),
    ("Black Pepper", "BLK PEPR"), ("Paprika", "PAPRIKA"),
    ("Cumin", "CUMIN"), ("Chicken Broth", "CHKN BRTH"),
    ("Vegetable Broth", "VEG BRTH"), ("Soy Sauce", "SOY SCE"),
    ("Hot Sauce", "HOT SCE"), ("Ketchup", "KTCHP"),
    ("Mayonnaise", "MAYO"), ("Mustard", "MUSTRD"),
    ("Salsa", "SALSA"), ("Coconut Milk", "CCN MLK"),
    ("Tomato Paste", "TOM PASTE"), ("Vinegar", "VINEGAR"),
]

FROZEN = [
    ("Frozen Pizza", "FRZN PZA"), ("Frozen Vegetables", "FRZN VEG"),
    ("Frozen Berries", "FRZN BRY"), ("Frozen Peas", "FRZN PEAS"),
    ("Frozen Corn", "FRZN CORN"), ("Frozen Spinach", "FRZN SPNCH"),
    ("Ice Cream", "ICE CRM"), ("Frozen Chicken Nuggets", "FRZN NGT"),
    ("Frozen Waffles", "FRZN WFFL"), ("Frozen Burritos", "FRZN BRTO"),
    ("Frozen Fish Sticks", "FRZN FSH"), ("Frozen Edamame", "FRZN EDAME"),
]

SNACKS = [
    ("Potato Chips", "PTTO CHIP"), ("Tortilla Chips", "TRT CHIP"),
    ("Pretzels", "PRTZL"), ("Popcorn", "POPCORN"),
    ("Granola Bars", "GRNLA BAR"), ("Protein Bars", "PRTN BAR"),
    ("Mixed Nuts", "MXD NUTS"), ("Trail Mix", "TRL MIX"),
    ("Rice Cakes", "RICE CAKE"), ("Crackers", "CRKRS"),
    ("Graham Crackers", "GRHM CRKR"), ("Beef Jerky", "BEEF JRKY"),
    ("Sunflower Seeds", "SNFL SDS"), ("Pumpkin Seeds", "PMPKN SDS"),
]

BEVERAGES = [
    ("Orange Juice", "OJ"), ("Apple Juice", "APL JCE"),
    ("Cranberry Juice", "CRN JCE"), ("Grape Juice", "GRP JCE"),
    ("Lemonade", "LMNADE"), ("Coffee", "COFFEE"),
    ("Ground Coffee", "GRD COF"), ("Instant Coffee", "INST COF"),
    ("Tea Bags", "TEA BAG"), ("Green Tea", "GRN TEA"),
    ("Sparkling Water", "SPRK WTR"), ("Bottled Water", "BTL WTR"),
    ("Cola", "COLA"), ("Diet Cola", "DT COLA"),
    ("Sports Drink", "SPRT DRK"), ("Energy Drink", "ENRGY DRK"),
    ("Coconut Water", "CCN WTR"),
]

BREAKFAST = [
    ("Oatmeal", "OATMEAL"), ("Rolled Oats", "RLD OATS"),
    ("Granola", "GRNLA"), ("Pancake Mix", "PNCK MX"),
    ("Waffle Mix", "WFL MX"), ("Maple Syrup", "MPLE SYP"),
    ("Cereal", "CEREAL"),
]

CONDIMENTS = [
    ("Bbq Sauce", "BBQ SCE"), ("Ranch Dressing", "RNCH DRS"),
    ("Caesar Dressing", "CZAR DRS"), ("Italian Dressing", "ITL DRS"),
    ("Teriyaki Sauce", "TRYKI SCE"), ("Worcestershire Sauce", "WSTR SCE"),
    ("Relish", "RELISH"), ("Pickle", "PICKLE"),
]

PAKISTANI_BRANDS = [
    ("Olpers Milk", "OLPERS"), ("Olpers Cream", "OLPERS CRM"),
    ("Milkpak Milk", "MILKPAK"), ("Tarang Tea Whitener", "TARANG"),
    ("Nestle Everyday", "EVERYDAY"), ("Nestle Fruita Vitals", "FRTA VTL"),
    ("Nurpur Butter", "NURPUR"), ("Nurpur Cheese", "NURPUR CH"),
    ("Haleeb Milk", "HALEEB"), ("Prema Milk", "PREMA"),
    ("Anhaar Milk", "ANHAAR"), ("Good Milk", "GOOD MLK"),
    ("Shan Biryani Masala", "SHAN BRYNI"), ("Shan Karahi Masala", "SHAN KRHI"),
    ("Shan Qorma Masala", "SHAN QRMA"), ("Shan Nihari Masala", "SHAN NHARI"),
    ("National Biryani Masala", "NAT BRYNI"), ("National Ketchup", "NAT KTCHP"),
    ("National Pickle", "NAT PICKLE"), ("Laziza Biryani Masala", "LAZIZA"),
    ("Shangrila Ketchup", "SHNGRLA"), ("Shangrila Chili Sauce", "SHNGRLA CHL"),
    ("Mitchells Jam", "MTCHLLS"), ("Mitchells Candy", "MTCH CNDY"),
    ("Fauji Corn Flakes", "FAUJI CF"), ("Fauji Muesli", "FAUJI MUSL"),
    ("Nido Milk Powder", "NIDO"), ("Milo", "MILO"),
    ("Rooh Afza", "ROOH AFZ"), ("Tang", "TANG"),
    ("Knorr Noodles", "KNORR NDL"), ("Knorr Chicken Powder", "KNORR CHKN"),
    ("Peek Freans Biscuits", "PEEK FRN"), ("Sooper Biscuits", "SOOPER"),
    ("Rio Biscuits", "RIO"), ("Prince Biscuits", "PRINCE"),
]

GLOBAL_BRANDS = [
    ("Nutella", "NUTELLA"), ("Kit Kat", "KITKAT"),
    ("Snickers", "SNCKRS"), ("Twix", "TWIX"),
    ("Mars Bar", "MARS"), ("Bounty", "BOUNTY"),
    ("M and Ms", "MNM"), ("Milky Way", "MLKYWAY"),
    ("Toblerone", "TOBLRN"), ("Cadbury Dairy Milk", "CDBRY"),
    ("Hersheys Chocolate Bar", "HRSHY BAR"),
    ("Oreo Cookies", "OREO"), ("Chips Ahoy", "CHPSAHOY"),
    ("Pringles", "PRNGLS"), ("Doritos", "DORITOS"),
    ("Lays Chips", "LAYS"), ("Cheetos", "CHEETOS"),
    ("Ruffles", "RUFFLES"), ("Sun Chips", "SNCHIPS"),
    ("Coca Cola", "COKE"), ("Pepsi", "PEPSI"),
    ("Sprite", "SPRITE"), ("Fanta", "FANTA"),
    ("Mountain Dew", "MTN DEW"), ("7 Up", "7UP"),
    ("Mirinda", "MIRINDA"), ("Red Bull", "REDBULL"),
    ("Monster Energy", "MONSTER"), ("Gatorade", "GATORADE"),
    ("Nescafe Coffee", "NESCAFE"), ("Lipton Tea", "LIPTON"),
    ("Kelloggs Corn Flakes", "CRNFLKS"), ("Cheerios", "CHEERIOS"),
    ("Lucky Charms", "LCKY CHRM"), ("Special K", "SPCL K"),
    ("Pop Tarts", "POPTART"), ("Nature Valley Bars", "NTR VLY"),
    ("Quaker Oats", "QKROATS"), ("Ben and Jerrys Ice Cream", "BNJRY"),
    ("Haagen Dazs", "HDZS"), ("Heinz Ketchup", "HEINZ"),
    ("Hellmanns Mayo", "HLMNS"), ("Kraft Singles", "KRFT SNG"),
]

ALL_FOOD_ITEMS = (
    PRODUCE + DAIRY + MEAT + SEAFOOD + BAKERY + PANTRY +
    FROZEN + SNACKS + BEVERAGES + BREAKFAST + CONDIMENTS +
    PAKISTANI_BRANDS + GLOBAL_BRANDS
)

# ── Receipt generation vocabulary ─────────────────────────────────────────────

UNITS = ["LB", "OZ", "GAL", "GL", "CT", "PK", "EA", "BAG", "BTL", "BX", "CAN", "DOZ", "PT", "QT"]
PRICE_RANGE = (0.49, 24.99)

MODIFIERS = [
    "", "", "", "",          # empty weighted 4x so most lines have no modifier
    "ORG", "ORGANIC",
    "FMLY PK", "VALUE PK",
    "PREM", "LG", "XLG",
    "BNLS", "FRESH", "FRZN",
    "LOW FAT", "FF",        # fat-free
]

TAX_CODES  = ["", "", "", "TAX A", "TAX B", "T", "F", "N"]   # mostly empty
DEPT_CODES = ["", "", "", "#1234", "#5678", "#9012", "D1", "D4"]

# ── Variant generator ─────────────────────────────────────────────────────────

def get_food_variant(food_abbr: str) -> str:
    """
    Apply a random modifier and/or structural corruption to a food abbreviation.
    Mirrors the messy, inconsistent token sequences seen on real receipts.
    """
    modifier = random.choice(MODIFIERS)

    # Base candidates: original, no-space, hyphen, underscore
    base_candidates = [food_abbr]
    if " " in food_abbr:
        base_candidates += [
            food_abbr.replace(" ", ""),
            food_abbr.replace(" ", "-"),
            food_abbr.replace(" ", "_"),
        ]

    base = random.choice(base_candidates)

    if modifier:
        # Modifier can be prepended with or without a space
        sep = "" if random.random() < 0.15 else " "
        return f"{modifier}{sep}{base}"
    return base

# ── OCR noise ─────────────────────────────────────────────────────────────────

NOISE_MAP = {
    "0": "O", "O": "0",
    "1": "I", "I": "1",
    "5": "S", "S": "5",
    "8": "B", "B": "8",
    "2": "Z", "Z": "2",
    "l": "1", "G": "6",
}

def add_ocr_noise(tokens: list[str], prob: float = 0.06) -> list[str]:
    """
    Simulate common TrOCR output errors at character and token level.
    char-level : random substitutions from NOISE_MAP
    token-level: occasional concatenation, trailing punctuation, separator swap
    """
    noisy = []
    for token in tokens:
        # Character-level substitution
        chars = list(token)
        for i, c in enumerate(chars):
            if random.random() < prob and c in NOISE_MAP:
                chars[i] = NOISE_MAP[c]
        token = "".join(chars)

        # Token-level structural corruption (low probability each)
        r = random.random()
        if r < 0.02:
            token = token.rstrip() + "."      # trailing period
        elif r < 0.04:
            token = token.rstrip() + ","      # trailing comma
        elif r < 0.05:
            token = token.replace(" ", "")    # merged token

        noisy.append(token)

    # Occasionally merge two adjacent tokens (simulates OCR over-joining)
    if len(noisy) >= 2 and random.random() < 0.03:
        idx = random.randrange(len(noisy) - 1)
        noisy[idx] = noisy[idx] + noisy[idx + 1]
        del noisy[idx + 1]

    return noisy

# ── Receipt line generator ────────────────────────────────────────────────────

def generate_synthetic_line() -> dict:
    """
    Generate one synthetic grocery receipt line with BIO annotations.
    Covers 20 distinct receipt layout styles found in real-world receipts.
    """
    food_full, base_abbr = random.choice(ALL_FOOD_ITEMS)
    food_abbr = get_food_variant(base_abbr)
    food_tokens = food_abbr.split()
    full_tokens = food_full.upper().split()

    qty   = random.randint(1, 6)
    unit  = random.choice(UNITS)
    price = round(random.uniform(*PRICE_RANGE), 2)
    sale  = round(price - random.uniform(0.5, 3.0), 2)
    multi = qty * 2

    n_food      = len(food_tokens)
    n_full      = len(full_tokens)
    tax_code    = random.choice(TAX_CODES)
    dept_code   = random.choice(DEPT_CODES)

    def food_tags(n):
        return [LABEL2ID["B-FOOD"]] + [LABEL2ID["I-FOOD"]] * (n - 1)

    style = random.randint(0, 19)

    # ── style 0 : "STRWBRY 1 LB 2.99"
    if style == 0:
        tokens = food_tokens + [str(qty), unit, str(price)]
        tags   = food_tags(n_food) + [LABEL2ID["B-QTY"], LABEL2ID["B-UNIT"], LABEL2ID["B-PRICE"]]

    # ── style 1 : "ORG STRWBRY 1LB 2.99"
    elif style == 1:
        combined = f"{qty}{unit}"
        tokens   = food_tokens + [combined, str(price)]
        tags     = food_tags(n_food) + [LABEL2ID["B-QTY"], LABEL2ID["B-PRICE"]]

    # ── style 2 : "STRAWBERRIES 1 LB"  (full name, no price)
    elif style == 2:
        tokens = full_tokens + [str(qty), unit]
        tags   = food_tags(n_full) + [LABEL2ID["B-QTY"], LABEL2ID["B-UNIT"]]

    # ── style 3 : "2 CHKN BRST 5.99"
    elif style == 3:
        tokens = [str(qty)] + food_tokens + [str(price)]
        tags   = [LABEL2ID["B-QTY"]] + food_tags(n_food) + [LABEL2ID["B-PRICE"]]

    # ── style 4 : "CHKN BRST EA 5.99"  (unit before price, no qty)
    elif style == 4:
        tokens = food_tokens + [unit, str(price)]
        tags   = food_tags(n_food) + [LABEL2ID["B-UNIT"], LABEL2ID["B-PRICE"]]

    # ── style 5 : "CHKN BRST 5.99"  (no qty, no unit)
    elif style == 5:
        tokens = food_tokens + [str(price)]
        tags   = food_tags(n_food) + [LABEL2ID["B-PRICE"]]

    # ── style 6 : "CHKN BRST @5.99/LB"  (price-per-unit notation)
    elif style == 6:
        price_token = f"@{price}/{unit}"
        tokens = food_tokens + [str(qty), price_token]
        tags   = food_tags(n_food) + [LABEL2ID["B-QTY"], LABEL2ID["B-PRICE"]]

    # ── style 7 : "2X CHKN BRST 10.98"  (multiplier prefix)
    elif style == 7:
        tokens = [f"{qty}X"] + food_tokens + [str(round(price * qty, 2))]
        tags   = [LABEL2ID["B-QTY"]] + food_tags(n_food) + [LABEL2ID["B-PRICE"]]

    # ── style 8 : "CHKN BRST SAVE 1.00"  (discount line, no std price)
    elif style == 8:
        discount = round(random.uniform(0.25, 3.00), 2)
        tokens   = food_tokens + ["SAVE", str(discount)]
        tags     = food_tags(n_food) + [LABEL2ID["O"], LABEL2ID["B-PRICE"]]

    # ── style 9 : "CHKN BRST SALE 4.99"  (sale price label)
    elif style == 9:
        tokens = food_tokens + ["SALE", str(sale)]
        tags   = food_tags(n_food) + [LABEL2ID["O"], LABEL2ID["B-PRICE"]]

    # ── style 10 : "CHKN BRST REG 5.99 SALE 4.49"  (reg + sale)
    elif style == 10:
        tokens = food_tokens + ["REG", str(price), "SALE", str(sale)]
        tags   = food_tags(n_food) + [LABEL2ID["O"], LABEL2ID["B-PRICE"],
                                       LABEL2ID["O"], LABEL2ID["B-PRICE"]]

    # ── style 11 : "CHKN BRST TAX A 5.99"  (with tax code)
    elif style == 11:
        tc     = tax_code if tax_code else "TAX A"
        tokens = food_tokens + tc.split() + [str(price)]
        tags   = food_tags(n_food) + [LABEL2ID["O"]] * len(tc.split()) + [LABEL2ID["B-PRICE"]]

    # ── style 12 : "CHKN BRST #12345 5.99"  (with dept/SKU code)
    elif style == 12:
        dc     = dept_code if dept_code else "#12345"
        tokens = food_tokens + [dc, str(price)]
        tags   = food_tags(n_food) + [LABEL2ID["O"], LABEL2ID["B-PRICE"]]

    # ── style 13 : "CHKN BRST BOGO"  (buy-one-get-one, no price token)
    elif style == 13:
        tokens = food_tokens + ["BOGO"]
        tags   = food_tags(n_food) + [LABEL2ID["O"]]

    # ── style 14 : "CHKN BRST DISC"  (discount marker only)
    elif style == 14:
        tokens = food_tokens + ["DISC"]
        tags   = food_tags(n_food) + [LABEL2ID["O"]]

    # ── style 15 : "3 @ 1.99 CHKN BRST"  (qty + unit-price then food)
    elif style == 15:
        tokens = [str(qty), "@", str(price)] + food_tokens
        tags   = [LABEL2ID["B-QTY"], LABEL2ID["O"], LABEL2ID["B-PRICE"]] + food_tags(n_food)

    # ── style 16 : "CHKN BRST 1 LB @2.99/LB"  (full qty + per-unit)
    elif style == 16:
        price_token = f"@{price}/{unit}"
        tokens = food_tokens + [str(qty), unit, price_token]
        tags   = food_tags(n_food) + [LABEL2ID["B-QTY"], LABEL2ID["B-UNIT"], LABEL2ID["B-PRICE"]]

    # ── style 17 : "CHKN BRST PK 3 5.99"  (pack unit before qty)
    elif style == 17:
        tokens = food_tokens + ["PK", str(qty), str(price)]
        tags   = food_tags(n_food) + [LABEL2ID["B-UNIT"], LABEL2ID["B-QTY"], LABEL2ID["B-PRICE"]]

    # ── style 18 : "STRAWBERRIES"  (name only — very short receipt line)
    elif style == 18:
        tokens = full_tokens
        tags   = food_tags(n_full)

    # ── style 19 : "CHKN BRST 1LB TAX A 5.99"  (combined qty-unit + tax + price)
    else:
        combined = f"{qty}{unit}"
        tc       = tax_code if tax_code else "T"
        tokens   = food_tokens + [combined] + tc.split() + [str(price)]
        tags     = (food_tags(n_food) +
                    [LABEL2ID["B-QTY"]] +
                    [LABEL2ID["O"]] * len(tc.split()) +
                    [LABEL2ID["B-PRICE"]])

    assert len(tokens) == len(tags), (
        f"Token/tag length mismatch in style {style}: "
        f"{len(tokens)} tokens vs {len(tags)} tags\n"
        f"tokens={tokens}\ntags={tags}"
    )
    return {"tokens": tokens, "ner_tags": tags}


# ── Dataset generator ─────────────────────────────────────────────────────────

def generate_synthetic_dataset(n: int = 5000) -> list[dict]:
    examples = []
    for _ in range(n):
        ex = generate_synthetic_line()
        ex["tokens"] = add_ocr_noise(ex["tokens"])
        examples.append(ex)
    print(f"Synthetic: {len(examples)} lines generated "
          f"from {len(ALL_FOOD_ITEMS)} food items across 20 styles")
    return examples

### Combined Dataset Builder

In [ ]:
from sklearn.model_selection import train_test_split

def build_ner_dataset() -> dict:
    """
    Combine all sources, split into train/val/test.
    Weights: CORD 1x, TASTEset 1x, Synthetic 0.8x
    """
    cord_data      = load_cord_ner()
    tasteset_data  = load_tasteset_ner()
    synthetic_data = generate_synthetic_dataset(10000)

    # Apply synthetic weight (0.8x)
    synthetic_sample = random.sample(synthetic_data, int(len(synthetic_data) * 0.8))

    all_data = cord_data + tasteset_data + synthetic_sample
    random.shuffle(all_data)

    # 80/10/10 split
    train_val, test = train_test_split(all_data, test_size=0.10, random_state=42)
    train, val      = train_test_split(train_val, test_size=0.111, random_state=42)

    print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
    return {"train": train, "validation": val, "test": test}

ner_splits = build_ner_dataset()

### **Tokenization & Label Alignment**
DistilBERT uses WordPiece tokenization which splits words into subword tokens. Labels must be aligned: first subword gets the real label, continuation subwords get -100 (ignored in loss).

In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align(example: dict) -> dict:
    """
    Tokenize pre-split word list and align NER labels to subword tokens.

    Example:
      Input tokens:  ["STRWBRY",  "1",  "LB"]
      WordPiece:     ["str", "##wb", "##ry", "1", "lb"]
      Aligned tags:  [B-FOOD,     -100,  -100,  B-QTY, B-UNIT]
    """
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        max_length=128,
        is_split_into_words=True,   # CRITICAL: input is pre-tokenized word list
    )

    word_ids = tokenized.word_ids()
    labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)              # [CLS] / [SEP] tokens
        elif word_id != prev_word_id:
            labels.append(example["ner_tags"][word_id])  # first subword → real label
        else:
            labels.append(-100)              # continuation subwords → ignore
        prev_word_id = word_id

    tokenized["labels"] = labels
    return tokenized


def build_hf_dataset(split_data: list[dict]):
    """Convert list of examples to HuggingFace Dataset and tokenize."""
    from datasets import Dataset
    ds = Dataset.from_list(split_data)
    ds = ds.map(
        tokenize_and_align,
        remove_columns=["tokens", "ner_tags"],
    )
    return ds

train_ds = build_hf_dataset(ner_splits["train"])
val_ds   = build_hf_dataset(ner_splits["validation"])
test_ds  = build_hf_dataset(ner_splits["test"])

print(f"Tokenized — Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

### Model Setup

In [ ]:
from transformers import AutoModelForTokenClassification
from pathlib import Path

# Load from checkpoint if resuming, otherwise load base model
# Loading from checkpoint preserves fine-tuned weights — don't load base model then resume
if RESUME_FROM and Path(RESUME_FROM).exists():
    print(f"Loading model from checkpoint: {RESUME_FROM}")
    model = AutoModelForTokenClassification.from_pretrained(
        RESUME_FROM,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=False,
    )
else:
    print("Loading base model for fresh training")
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(LABEL2ID),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

### Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert-ner-smart-stock",

    # Training schedule
    num_train_epochs=15,
    per_device_train_batch_size=32,    # NER is much lighter than TrOCR — batch 32 fits fine
    per_device_eval_batch_size=32,

    # Optimizer
    learning_rate=3e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,

    # Eval & saving
    eval_strategy="epoch",

    # Adding these 2 lines
    save_strategy="no",
    load_best_model_at_end=False,
    
    # save_strategy="epoch",
    # load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    # save_total_limit=3,
    save_total_limit=2,   # Adding this line

    # Performance
    fp16=True,

    # Logging
    logging_steps=50,
    log_level="info",
    report_to="none",
)

 ### Evaluation — seqeval F1
 Token accuracy is misleading for NER — O tokens dominate and inflate it. Always use entity-level F1 via seqeval.

In [ ]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")
label_names = list(LABEL2ID.keys())

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = [
        [label_names[pred] for pred, lab in zip(prediction, label) if lab != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[lab] for lab in label if lab != -100]
        for label in labels
    ]

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": round(results["overall_precision"], 4),
        "recall":    round(results["overall_recall"],    4),
        "f1":        round(results["overall_f1"],        4),
        "accuracy":  round(results["overall_accuracy"],  4),
    }

### Data Collator

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    label_pad_token_id=-100,    # pads labels with -100 (ignored in loss)
)

### Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# RESUME_FROM defined in paths cell at top
# Trainer reads trainer_state.json from checkpoint — knows epochs already done
trainer.train(resume_from_checkpoint=RESUME_FROM)

### Save & Export

In [ ]:
from pathlib import Path

save_path = "/kaggle/working/distilbert-ner-smart-stock-best"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# Verify files
expected = [
    "model.safetensors", "config.json",
    "tokenizer_config.json", "tokenizer.json",
    "vocab.txt", "special_tokens_map.json",
]
print(f"\nSaved to: {save_path}")
for fname in expected:
    fpath = Path(save_path) / fname
    exists = fpath.exists()
    size = fpath.stat().st_size / 1e6 if exists else 0
    print(f"  {'✅' if exists else '❌'} {fname} ({size:.1f} MB)")

### Evaluate on Test Set

In [ ]:
results = trainer.evaluate(test_ds)
print(f"Test F1:        {results['eval_f1']:.4f}")
print(f"Test Precision: {results['eval_precision']:.4f}")
print(f"Test Recall:    {results['eval_recall']:.4f}")

# Target benchmarks:
# F1        ≥ 0.88
# Precision ≥ 0.90
# Recall    ≥ 0.86

### Entity Post-Processing (Inference)

In [ ]:
def extract_entities(tokens: list[str], predictions: list[str]) -> dict:
    """
    Group BIO predictions into entity spans.

    Input:
      tokens:      ["ORG", "STRWBRY", "1", "LB", "2.99"]
      predictions: ["B-FOOD", "I-FOOD", "B-QTY", "B-UNIT", "B-PRICE"]

    Output:
      {"food_tokens": ["ORG", "STRWBRY"], "quantity": "1", "unit": "LB", "price": "2.99"}
    """
    entities = {"food_tokens": [], "quantity": None, "unit": None, "price": None}

    for token, pred in zip(tokens, predictions):
        if pred in ("B-FOOD", "I-FOOD"):
            entities["food_tokens"].append(token)
        elif pred == "B-QTY":
            entities["quantity"] = token
        elif pred == "B-UNIT":
            entities["unit"] = token
        elif pred == "B-PRICE":
            entities["price"] = token

    return entities

### ONNX Export

In [ ]:
from optimum.exporters.onnx import main_export

main_export(
    model_name_or_path=save_path,
    output="./models/distilbert_ner_onnx/",
    task="token-classification",
    opset=18,  # minimum recommended for distilbert is 18 (14 causes warning)
)
# Rename output: distilbert_ner_onnx/model.onnx → distilbert_ner.onnx
# Matches ml_service/models/ structure from ML_Pipeline.md

#### ONNX inference:

In [ ]:
import onnxruntime as ort
import numpy as np

# Local path (during training session)
session = ort.InferenceSession("./models/distilbert_ner_onnx/model.onnx")
# From Kaggle dataset (future sessions)
# session = ort.InferenceSession("/kaggle/input/datasets/maazahmad69/distilbert-ner-onnx/model.onnx")

def run_ner_onnx(tokens: list[str]) -> list[str]:
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="np",
        max_length=128,
        padding="max_length",
        truncation=True,
    )
    logits = session.run(
        None,
        {
            "input_ids":      encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
        }
    )[0]
    pred_ids = np.argmax(logits, axis=-1)[0]
    word_ids = tokenizer(tokens, is_split_into_words=True).word_ids()

    # Align back to word level — keep only first subword prediction per word
    word_preds = {}
    for idx, wid in enumerate(word_ids):
        if wid is not None and wid not in word_preds:
            word_preds[wid] = ID2LABEL[pred_ids[idx]]

    return [word_preds[i] for i in range(len(tokens))]

### Optuna Hyperparameter Search

In [ ]:
!pip install optuna -q

def optuna_hp_space(trial):
    return {
        "learning_rate":               trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
        "warmup_ratio":                trial.suggest_float("warmup_ratio", 0.03, 0.15),
        "weight_decay":                trial.suggest_float("weight_decay", 0.0, 0.05),
    }

def model_init():
    return AutoModelForTokenClassification.from_pretrained(
        MODEL_CHECKPOINT,
        num_labels=len(LABEL2ID),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

search_train = train_ds.select(range(len(train_ds) // 5))

search_trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=search_train,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

best_run = search_trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=10,
    compute_objective=lambda metrics: metrics["eval_f1"],
)

print("Best hyperparameters:", best_run.hyperparameters)

# Apply best params for final full training run
# NOTE: Prefer batch=32 over batch=64 — batch=64 underconverges on small search subset
# If best trial used batch=64, consider overriding with batch=32
recommended = {
    "learning_rate": 2.4030906082694194e-05,
    "per_device_train_batch_size": 32,
    "warmup_ratio": 0.09203872502951205,
    "weight_decay": 0.014083492139621796,
}
for k, v in recommended.items():
    setattr(training_args, k, v)
print("Applied recommended hyperparameters:", recommended)